# 1. PKD YOLO26s → YOLO26n pada Poltrada V3

**Eksperimen 12 PKD** · Satu notebook standalone Kaggle, student YOLO26n, teacher **YOLO26s v62**.
Notebook lain, termasuk CSAKD bernomor 12, tidak diubah atau ditimpa. Seed pilot 42.

Default **`RUN_FULL_TRAINING=False`**: Run All menjalankan audit, numerical verification dan
**GPU diagnostic 20 batch pada 640/batch16**, lalu berhenti. Full training 100 epoch hanya
berjalan setelah diagnostic PASS dan flag eksplisit True. Tidak ada penurunan batch/resolusi otomatis.

Checkpoint inference `best.pt`/`last.pt` berisi student-only. Archive training per epoch
menyimpan student, projector, optimizer, scheduler, scaler dan RNG, dengan status
**`RESUME_NOT_VERIFIED`**. Pemulihan engine/worker belum didukung; jangan menganggap `last.pt`
sebagai jaminan resume. `resume=True` ditolak. Laporan lokal tidak mengklaim keberhasilan GPU.


## 2. Landasan PKD dan perbedaan dengan Native KD

Acuan: [Cao dkk., NeurIPS 2022](https://papers.neurips.cc/paper_files/paper/2022/file/631ad9ae3174bf4d6c0f6fdca77335a4-Paper-Conference.pdf)
dan [PKDLoss MMRazor v1.0.0](https://github.com/open-mmlab/mmrazor/blob/v1.0.0/mmrazor/models/losses/pkd_loss.py).
Rumus numerik sama dengan source `main` yang diaudit; release dan hash dikunci untuk verifikasi.

Untuk setiap channel fitur BCHW, hitung mean dan **sample standard deviation** pada B,H,W
(`correction=1`), lalu normalisasi dengan pembagi `std + 1e-6`.

\[
L_{PKD}=\sum_{i\in\{P3,P4,P5\}}\tfrac12\operatorname{mean}((\hat F_{S,i}-\hat F_{T,i})^2),
\quad L=L_{det}+\lambda_{PKD}L_{PKD}.
\]

Dengan sample std dan epsilon, hubungannya adalah pendekatan terhadap `1-r`; faktor
finite-sample `(m-1)/m` dan epsilon tidak dihilangkan. Statistik mencakup seluruh batch,
bukan dihitung terpisah per gambar atau di dalam chunk batch.

Adaptasi YOLO26: hanya tiga neck input Detect, proyeksi channel **satu Conv1×1 tanpa bias**
bila perlu, tanpa resize. PKD dan projector dihitung FP32 di luar autocast. Detection loss
one-to-many/one-to-one tetap resmi. KD mean dikalikan batch size tepat sekali pada kontrak trainer.
Native menggunakan score-weighted L2 dan projector berbeda; loss itu **tidak dipanggil** di sini.
Tidak ada CrossKD/CSAKD, attention matrix, atau forward teacher-head.

Lihat juga [panduan Native Ultralytics](https://docs.ultralytics.com/guides/knowledge-distillation/).
API yang dipakai diaudit dari paket lokal **Ultralytics 8.4.155**, karena link source tag
yang diminta tidak dapat diakses oleh alat browsing saat implementasi.


## 3. Konfigurasi penelitian

Ubah konfigurasi hanya di sini. PKD_WEIGHT=1.0 adalah pilot; tinjau rasio loss pada diagnostic.


In [ ]:
EXPERIMENT_ID = "12_pkd_yolo26n"
METHOD = "pkd"
DATA = "/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml"
MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
DEVICE = 0
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"
PROJECT = "/kaggle/working/results"
WANDB_ENABLED = True
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_GROUP = "teacher_s_student_n_pkd"
WANDB_TAGS = [METHOD, "teacher-s", "student-n", f"seed{SEED}", "poltrada-v3"]
PKD_WEIGHT = 1.0  # Pilot; ukur rasio PKD/detection sebelum menetapkan bobot final.
PKD_EPS = 1e-6
RUN_FULL_TRAINING = False
DIAGNOSTIC_BATCH = 16
DIAGNOSTIC_STEPS = 20
DIAGNOSTIC_WARMUP_STEPS = 5
MEMORY_GROWTH_TOL_MB = 256
EXPECTED_NC = 6
TEACHER_ARTIFACT = (
    "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/"
    "02_baseline_yolo26s-q5kp9k9d-checkpoints:v62"
)
TEACHER_RUN_ID = "q5kp9k9d"
TEACHER_CHECKPOINT = globals().get("TEACHER_CHECKPOINT", "")  # Diisi audit; tetap tersedia saat hanya flag full diubah.
REFERENCE_URL = "https://raw.githubusercontent.com/open-mmlab/mmrazor/v1.0.0/mmrazor/models/losses/pkd_loss.py"
REFERENCE_SHA256 = "41d342f58c56efdcadb188dfc6d77a492d05e1ec2bee11240cd2cff9230ed68c"
NUMERICAL_ATOL = 1e-6
NUMERICAL_RTOL = 1e-5
RESUME_STATUS = "RESUME_NOT_VERIFIED"


## 4. Lingkungan dan versi

Aktifkan satu GPU dan Internet Kaggle. PyTorch/CUDA bawaan dipertahankan. Jika paket diinstal ulang setelah import, restart kernel.


In [ ]:
import os
import subprocess
import sys
from importlib import metadata

for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"[ENV] Installing {package}=={version}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])

import ast
import base64
import copy
import gc
import hashlib
import json
import math
import random
import shutil
import statistics
import time
import types
import typing
import urllib.request
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from ultralytics import YOLO, settings
from ultralytics import __version__ as ultralytics_version
from ultralytics.cfg import get_cfg
from ultralytics.data.utils import check_det_dataset
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.distill_model import DistillationModel, FeatureHook
from ultralytics.nn.modules.head import Detect
from ultralytics.nn.tasks import DetectionModel, load_checkpoint
from ultralytics.utils.torch_utils import (
    copy_attr,
    get_flops,
    init_seeds,
    strip_optimizer,
    unwrap_model,
)

import wandb

if ultralytics_version != "8.4.155" or wandb.__version__ != "0.28.1":
    raise RuntimeError("Restart kernel setelah instalasi: Ultralytics harus 8.4.155.")
settings.update({"wandb": False})
LAST_PHASE = {"value": "initialization"}


def status(stage: str, message: str) -> None:
    print(f"[{datetime.now().astimezone():%H:%M:%S}] [{stage}] {message}", flush=True)


def require_cuda() -> None:
    if not torch.cuda.is_available() or DEVICE != 0:
        raise RuntimeError("Aktifkan satu GPU CUDA Kaggle, DEVICE=0. GPU diagnostic tidak boleh diganti CPU.")


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def state_digest(model: nn.Module) -> str:
    """Hash parameters AND buffers, including teacher BatchNorm statistics."""
    digest = hashlib.sha256()
    for key, value in model.state_dict().items():
        digest.update(key.encode())
        digest.update(value.detach().cpu().contiguous().reshape(-1).view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()


def git_commit() -> str | None:
    try:
        result = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=False)
        return result.stdout.strip() if result.returncode == 0 else None
    except OSError:
        return None


ENVIRONMENT_INFO = {
    "python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda,
    "ultralytics": metadata.version("ultralytics"), "wandb": metadata.version("wandb"),
    "git_commit": git_commit(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
SESSION_DIR = Path(PROJECT) / EXPERIMENT_ID / datetime.now(UTC).strftime("%Y%m%dT%H%M%S_%fZ")
SESSION_DIR.mkdir(parents=True, exist_ok=False)
status("ENV", json.dumps(ENVIRONMENT_INFO))


## 5. Autentikasi W&B

Tambahkan Secret WANDB_API_KEY dan aktifkan akses notebook. Token tidak disimpan di source atau output.


In [ ]:
require_cuda()
if not WANDB_ENABLED:
    raise ValueError("W&B online wajib untuk audit dan perlindungan checkpoint per epoch.")
if not os.environ.get("WANDB_API_KEY"):
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=os.environ["WANDB_API_KEY"])
if wandb.run is not None:
    raise RuntimeError("Ada run W&B aktif; selesaikan run itu atau restart kernel terlebih dahulu.")


## 6. Dataset

Dataset hanya dibaca; label diperiksa sesuai urutan indeks. Val dipakai untuk pemilihan checkpoint; test tidak digunakan untuk tuning.


In [ ]:
def class_names(names: dict | list) -> list[str]:
    return ([str(names[key]) for key in sorted(names, key=lambda key: int(key))]
            if isinstance(names, dict) else [str(name) for name in names])


if EVAL_SPLIT != "val":
    raise ValueError("Pemilihan checkpoint dan evaluasi pilot hanya memakai val.")
if not Path(DATA).is_file():
    raise FileNotFoundError(f"Dataset Kaggle tidak ditemukan: {DATA}")
data_info = check_det_dataset(DATA, autodownload=False)
if int(data_info["nc"]) != EXPECTED_NC or not data_info.get("train") or not data_info.get("val"):
    raise ValueError("Poltrada V3 harus memiliki train/val dan enam kelas.")
DATASET_NAMES = class_names(data_info["names"])
status("DATA", f"{DATA} | kelas={DATASET_NAMES}")


## 7. Download dan audit teacher

Artifact baseline YOLO26s tetap q5kp9k9d:v62, sama dengan Native 10/CrossKD 11, diunduh ke /kaggle/working/teacher_yolo26s_v62. Versi bukan nomor epoch. Tidak ada fallback teacher pretrained.


In [ ]:
def download_teacher() -> tuple[str, dict[str, Any]]:
    """Use only the fixed s/v62 artifact, validating source, bytes, schema and labels."""
    api = wandb.Api()
    artifact = api.artifact(TEACHER_ARTIFACT, type="model")
    if artifact.version != "v62" or artifact.name.split(":")[0] != "02_baseline_yolo26s-q5kp9k9d-checkpoints":
        raise ValueError("Teacher harus koleksi YOLO26s q5kp9k9d:v62.")
    producer = artifact.logged_by()
    if producer is None or producer.id != TEACHER_RUN_ID or list(producer.path) != [WANDB_ENTITY, WANDB_PROJECT, TEACHER_RUN_ID]:
        raise ValueError("Producer artifact bukan eksperimen teacher yang ditetapkan.")
    entry = artifact.manifest.entries.get("best.pt")
    if entry is None:
        raise ValueError("Manifest artifact tidak berisi best.pt; cache lokal bukan fallback.")
    status("TEACHER", f"Mengunduh {TEACHER_ARTIFACT}")
    directory = Path(artifact.download(root="/kaggle/working/teacher_yolo26s_v62"))
    checkpoint = directory / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"best.pt tidak tersedia: {checkpoint}")
    payload = checkpoint.read_bytes()
    if entry.size is None or len(payload) != entry.size:
        raise ValueError("Ukuran best.pt berbeda dari manifest.")
    if base64.b64encode(hashlib.md5(payload).digest()).decode() != entry.digest:
        raise ValueError("Checksum best.pt berbeda dari manifest.")
    digest = hashlib.sha256(payload).hexdigest()
    del payload
    teacher = reference = None
    try:
        teacher = YOLO(str(checkpoint))
        family = Path(str(teacher.model.yaml.get("yaml_file", "")).replace("\\", "/")).stem
        if family not in {"yolo26", "yolo26s"} or teacher.model.yaml.get("scale") != "s":
            raise ValueError("Checkpoint bukan YOLO26s.")
        if int(teacher.model.model[-1].nc) != EXPECTED_NC or class_names(teacher.model.names) != DATASET_NAMES:
            raise ValueError("Jumlah atau urutan label teacher berbeda dari dataset.")
        reference = DetectionModel("yolo26s.yaml", nc=EXPECTED_NC, verbose=False)
        if {k: tuple(v.shape) for k, v in teacher.model.state_dict().items()} != {
            k: tuple(v.shape) for k, v in reference.state_dict().items()
        }:
            raise ValueError("Schema tensor checkpoint tidak cocok dengan YOLO26s enam kelas.")
        artifact_metadata = dict(artifact.metadata or {})
        producer_config = dict(producer.config or {})
        checkpoint_args = dict((teacher.ckpt or {}).get("train_args") or {})
        declarations = {}
        for source, values in (("artifact", artifact_metadata), ("producer", producer_config), ("checkpoint", checkpoint_args)):
            dataset = values.get("dataset", values.get("data"))
            model = values.get("model")
            if dataset and str(dataset).replace("\\", "/") != DATA.replace("\\", "/"):
                raise ValueError(f"Dataset teacher berbeda pada {source}: {dataset}")
            if model and Path(str(model).replace("\\", "/")).name != "yolo26s.pt":
                raise ValueError(f"Deklarasi arsitektur teacher berbeda pada {source}: {model}")
            declarations[source] = {"dataset": dataset, "model": model}
        if not producer_config.get("dataset") or not producer_config.get("model") or not checkpoint_args.get("data"):
            raise ValueError("Provenance dataset/model teacher belum lengkap.")
        if not artifact.digest or not isinstance(artifact_metadata.get("epoch"), int):
            raise ValueError("Digest atau metadata epoch artifact belum tersedia; v62 bukan epoch.")
        teacher.model.eval().requires_grad_(False)
        provenance = {
            "requested_artifact": TEACHER_ARTIFACT, "resolved_name": artifact.name,
            "version": artifact.version, "artifact_digest": artifact.digest, "manifest_digest": entry.digest,
            "best_pt_sha256": digest, "checkpoint_path": str(checkpoint),
            "producer_id": producer.id, "producer_path": list(producer.path),
            "artifact_metadata": artifact_metadata, "declarations": declarations,
            "architecture": "yolo26s", "tensor_schema_verified": True, "nc": EXPECTED_NC,
            "names": DATASET_NAMES, "metadata_complete": True,
            "verified_at_utc": datetime.now(UTC).isoformat(),
        }
        status("TEACHER", f"{checkpoint} | artifact={artifact.name} | epoch={artifact_metadata.get('epoch')}")
        return str(checkpoint), provenance
    finally:
        teacher = reference = None
        gc.collect()


In [ ]:
TEACHER_CHECKPOINT, TEACHER_PROVENANCE = download_teacher()
(SESSION_DIR / "teacher_provenance.json").write_text(json.dumps(TEACHER_PROVENANCE, indent=2), encoding="utf-8")


## 8. Inisialisasi student

Setiap mode membuat student baru dari yolo26n.pt setelah reseed. Student diagnostic dibuang sebelum full training; diagnostic tidak menjadi warm start penelitian. Teacher frozen/eval dan dikeluarkan dari optimizer.


## 9. Feature taps dan alignment

P3/P4/P5 ditemukan dari Detect.f dan diverifikasi stride 8/16/32. Pada graph s/n yang diuji, koneksinya [16,19,22]. Channel dilihat lewat dry forward 64px saat konstruksi; diagnostic menguji bentuk aktual 640/batch16. Satu traversal teacher neck per batch, tanpa head; satu forward student dengan head resmi.


In [ ]:
def neck_taps(model: nn.Module) -> list[int]:
    """Discover ordered P3/P4/P5 from the actual Detect connections and strides."""
    heads = [layer for layer in model.model if isinstance(layer, Detect)]
    if len(heads) != 1 or not isinstance(heads[0].f, (list, tuple)) or len(heads[0].f) != 3:
        raise ValueError("Model harus memiliki satu Detect dengan tiga input neck.")
    head = heads[0]
    if [float(value) for value in head.stride] != [8.0, 16.0, 32.0]:
        raise ValueError("Urutan stride harus P3/P4/P5 = 8/16/32.")
    indices = list(head.f)
    if any(not isinstance(index, int) or not 0 <= index < head.i for index in indices):
        raise ValueError("Koneksi neck Detect tidak kompatibel.")
    return indices


def teacher_neck_forward(model: nn.Module, image: torch.Tensor, last_index: int) -> None:
    """One graph traversal up to the final neck tap; never execute the teacher Detect head."""
    saved, value = [], image
    for layer in model.model[:last_index + 1]:
        if layer.f != -1:
            value = (saved[layer.f] if isinstance(layer.f, int)
                     else [value if index == -1 else saved[index] for index in layer.f])
        value = layer(value)
        saved.append(value if layer.i in model.save else None)


## 10. Implementasi loss

Projector terdaftar sebagai submodule trainable. Semua cache hooks dibersihkan melalui finally setelah loss, termasuk jika terjadi error; target teacher tidak menerima autograd.


In [ ]:
def pkd_normalize(feature: torch.Tensor) -> torch.Tensor:
    """Per-channel sample std over B,H,W (correction=1); epsilon AFTER std, matching MMRazor."""
    if feature.ndim != 4 or feature.shape[0] * feature.shape[2] * feature.shape[3] <= 1:
        raise ValueError("PKD memerlukan BCHW dengan B*H*W > 1.")
    with torch.autocast(device_type=feature.device.type, enabled=False):
        value = feature.float()
        std, mean = torch.std_mean(value, dim=(0, 2, 3), correction=1, keepdim=True)
        normalized = (value - mean) / (std + PKD_EPS)
    if not torch.isfinite(normalized).all():
        raise RuntimeError("Fitur PKD ternormalisasi mengandung NaN/Inf.")
    return normalized


def pkd_feature_loss(student: torch.Tensor, teacher: torch.Tensor) -> torch.Tensor:
    """One level: half mean squared error of normalized, channel-aligned features."""
    if student.shape != teacher.shape:
        raise ValueError("Fitur PKD harus identik BCHW setelah alignment; resize tidak diaktifkan.")
    with torch.autocast(device_type=student.device.type, enabled=False):
        return 0.5 * F.mse_loss(pkd_normalize(student), pkd_normalize(teacher.detach()), reduction="mean")


class PKDModel(DistillationModel):
    """Native detection loss plus Pearson feature loss; no native score-weighted KD."""
    def __init__(self, teacher_model: str | Path | nn.Module, student_model: nn.Module):
        # Do not allocate Native KD's two-convolution/ReLU projector or capture head scores.
        nn.Module.__init__(self)
        if isinstance(teacher_model, (str, Path)):
            teacher_model = load_checkpoint(teacher_model)[0]
        device = next(student_model.parameters()).device
        self.teacher_model = teacher_model.to(device).float()
        self.student_model = student_model
        self._freeze_teacher()
        if class_names(teacher_model.names) != class_names(student_model.names):
            raise ValueError("Urutan label teacher dan student aktual berbeda.")
        self.feats_idx = neck_taps(student_model)
        if neck_taps(teacher_model) != self.feats_idx:
            raise ValueError("Koneksi neck teacher/student berbeda; adaptasi ini memerlukan graph YOLO26 yang sama.")
        self._teacher_feats, self._student_feats = {}, {}
        self._teacher_hooks, self._student_hooks = [], []
        self._register_feature_hooks()
        previous_training = student_model.training
        try:
            student_model.eval()
            with torch.no_grad():
                image = torch.zeros(2, student_model.yaml.get("channels", 3), 64, 64, device=device)
                teacher_neck_forward(teacher_model, image, max(self.feats_idx))
                student_model(image)
            projectors, layout = [], []
            for name, index in zip(("P3", "P4", "P5"), self.feats_idx):
                student, teacher = self._student_feats[index], self._teacher_feats[index]
                if student.shape[-2:] != teacher.shape[-2:]:
                    raise ValueError("Ukuran spasial neck berbeda; tidak ada interpolasi diam-diam.")
                cin, cout = student.shape[1], teacher.shape[1]
                projectors.append(nn.Identity() if cin == cout else nn.Conv2d(cin, cout, 1, bias=False))
                layout.append({"level": name, "index": index, "student_channels": cin,
                               "teacher_channels": cout, "projection": "identity" if cin == cout else "conv1x1-no-bias"})
            self.projector = nn.ModuleList(projectors).to(device)
            self.feature_layout = layout
            copy_attr(self, student_model)
            self.dis = 1.0  # Compatibility attribute only; Native KD loss is never invoked.
        except BaseException:
            self._remove_feature_hooks()
            raise
        finally:
            self._teacher_feats.clear()
            self._student_feats.clear()
            student_model.train(previous_training)
            self._freeze_teacher()

    def forward(self, value: torch.Tensor | dict, *args: Any, **kwargs: Any):
        if isinstance(value, dict):
            return self.loss(value, *args, **kwargs)
        try:
            return self.student_model.predict(value, *args, **kwargs)
        finally:
            self._teacher_feats.clear()
            self._student_feats.clear()

    def loss(self, batch: dict, preds: Any = None):
        self._teacher_feats.clear()
        self._student_feats.clear()
        try:
            if self.training:
                self.last_actual_batch = int(batch["img"].shape[0])
                self.last_feature_shapes = {}
                LAST_PHASE["value"] = "teacher_forward"
                with torch.no_grad():
                    teacher_neck_forward(self.teacher_model, batch["img"], max(self.feats_idx))
                LAST_PHASE["value"] = "student_forward"
                preds = self.student_model(batch["img"])
            LAST_PHASE["value"] = "detection_loss"
            detection, items = self.student_model.loss(batch, preds)
            parts = {}
            if self.training:
                LAST_PHASE["value"] = "pkd_forward"
                with torch.autocast(device_type=batch["img"].device.type, enabled=False):
                    for position, (level, index) in enumerate(zip(("P3", "P4", "P5"), self.feats_idx)):
                        student = self.projector[position](self._student_feats[index].float())
                        teacher = self._teacher_feats[index]
                        self.last_feature_shapes[level] = {
                            "student": list(self._student_feats[index].shape),
                            "teacher": list(teacher.shape), "aligned": list(student.shape)}
                        parts[level] = pkd_feature_loss(student, teacher)
            else:
                parts = {level: detection.new_zeros(()) for level in ("P3", "P4", "P5")}
            raw = sum(parts.values())
            weighted = PKD_WEIGHT * raw
            if not torch.isfinite(detection).all() or not torch.isfinite(weighted):
                raise RuntimeError("Detection/PKD loss non-finite; proses dihentikan.")
            batch_size = batch["img"].shape[0]
            per_image_detection = detection.detach().sum() / batch_size
            items.update({f"pkd_{level}": value.detach() for level, value in parts.items()})
            items.update(pkd_loss=raw.detach(), pkd_weighted_loss=weighted.detach(),
                         det_total_loss=per_image_detection,
                         total_loss=per_image_detection + weighted.detach(),
                         pkd_to_det_ratio=weighted.detach() / per_image_detection.abs().clamp_min(1e-12))
            LAST_PHASE["value"] = "backward"
            # Native detection is already batch-scaled. Apply B exactly ONCE to the KD mean.
            return torch.cat((detection.reshape(-1), (weighted * batch_size).reshape(1))), items
        finally:
            self._teacher_feats.clear()
            self._student_feats.clear()


## 11. Numerical verification terhadap MMRazor

Source release resmi diverifikasi SHA-256 sebelum mengeksekusi hanya class PKDLoss (decorator registry dilepas). Tidak memerlukan mmcv/mmengine. Perbandingan loss FP32 memakai atol 1e-6, rtol 1e-5; gradien memakai 1e-4. Kasus random, variansi kecil, konstan, identik dan anticorrelated diuji. Ini bukan tes GPU penelitian.


In [ ]:
def load_mmrazor_reference(source: str | None = None) -> tuple[nn.Module, str]:
    """Load only PKDLoss from the hash-checked official release; no mmcv/mmengine required."""
    if source is None:
        with urllib.request.urlopen(REFERENCE_URL, timeout=30) as response:
            source = response.read().decode("utf-8")
    source = source.rstrip() + "\n"
    digest = hashlib.sha256(source.encode()).hexdigest()
    if digest != REFERENCE_SHA256:
        raise ValueError("Hash kode referensi MMRazor berbeda; tinjau versi sebelum melanjutkan.")
    definition = next(node for node in ast.parse(source).body if isinstance(node, ast.ClassDef) and node.name == "PKDLoss")
    definition.decorator_list = []  # Omit only registry integration; numerical methods remain unchanged.
    namespace = {"torch": torch, "nn": nn, "F": F, "Tuple": tuple, "Union": typing.Union}
    # Execute only the class from the source whose canonical SHA-256 was checked above.
    exec(compile(ast.Module(body=[definition], type_ignores=[]), REFERENCE_URL, "exec"), namespace)  # noqa: S102
    return namespace["PKDLoss"](), digest


def numerical_verification(reference_source: str | None = None) -> dict[str, Any]:
    reference, digest = load_mmrazor_reference(reference_source)
    generator = torch.Generator().manual_seed(314159)
    cases, maximum_error = {}, 0.0
    for label, scale in (("random", 1.0), ("small_variance", 1e-8), ("constant", 0.0)):
        student = (torch.randn(3, 5, 9, 7, generator=generator) * scale).requires_grad_()
        teacher = torch.randn(3, 5, 9, 7, generator=generator) * scale
        actual, expected = pkd_feature_loss(student, teacher), reference(student, teacher)
        torch.testing.assert_close(actual, expected, atol=NUMERICAL_ATOL, rtol=NUMERICAL_RTOL)
        actual_gradient = torch.autograd.grad(actual, student, retain_graph=True)[0]
        expected_gradient = torch.autograd.grad(expected, student)[0]
        torch.testing.assert_close(actual_gradient, expected_gradient, atol=1e-4, rtol=1e-4)
        if not torch.isfinite(actual_gradient).all():
            raise RuntimeError("Gradien numerical test tidak finite.")
        error = float((actual - expected).detach().abs())
        maximum_error = max(maximum_error, error)
        cases[label] = {"loss": float(actual.detach()), "absolute_error": error}
    value = torch.randn(2, 4, 8, 8, generator=generator)
    identical = float(pkd_feature_loss(value, value))
    opposite = float(pkd_feature_loss(value, -value))
    if identical > NUMERICAL_ATOL or opposite <= identical + 0.1:
        raise RuntimeError("Uji identik/anticorrelated PKD gagal.")
    students = tuple(torch.randn(2, 3, size, size, generator=generator) for size in (8, 4, 2))
    teachers = tuple(torch.randn(2, 3, size, size, generator=generator) for size in (8, 4, 2))
    multi_actual = sum(pkd_feature_loss(student, teacher) for student, teacher in zip(students, teachers))
    multi_expected = reference(students, teachers)
    torch.testing.assert_close(multi_actual, multi_expected, atol=NUMERICAL_ATOL, rtol=NUMERICAL_RTOL)
    return {"status": "PASS", "reference_url": REFERENCE_URL, "reference_sha256": digest,
            "atol": NUMERICAL_ATOL, "rtol": NUMERICAL_RTOL, "cases": cases,
            "maximum_absolute_error": max(maximum_error, float((multi_actual - multi_expected).abs())),
            "identical_loss": identical, "opposite_loss": opposite, "three_level_sum_verified": True}


In [ ]:
NUMERICAL_RESULT = numerical_verification()
(SESSION_DIR / "numerical_verification.json").write_text(json.dumps(NUMERICAL_RESULT, indent=2), encoding="utf-8")
print(json.dumps(NUMERICAL_RESULT, indent=2))


## 12. GPU diagnostic: protokol, checkpoint, dan callback

Fungsi berikut disiapkan sebelum menjalankan diagnostic. Native optimizer/GradScaler tetap dipakai. Gradien diukur setelah native unscale/clipping melalui optimizer pre-hook. Overflow awal AMP boleh menjalani skip/backoff standar; langkah pengukuran terakhir harus stabil. Tidak ada unscale ganda atau perubahan initial scale manual.


In [ ]:
def protocol_args(directory: Path, *, diagnostic: bool = False) -> dict[str, Any]:
    """Same explicit training protocol as Native 05; other engine defaults remain pinned."""
    return dict(data=DATA, epochs=EPOCHS, imgsz=IMGSZ,  # noqa: C408 -- Keep reference training kwargs easy to compare.
                batch=DIAGNOSTIC_BATCH if diagnostic else BATCH,
                optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
                workers=WORKERS, patience=PATIENCE, amp=AMP,
                weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS,
                close_mosaic=CLOSE_MOSAIC, project=str(directory), name="training", exist_ok=False,
                plots=not diagnostic, pretrained=True, save=True,
                distill_model=str(TEACHER_CHECKPOINT), dis=1.0, compile=False, resume=False)


def audit_optimizer(model: PKDModel, optimizer: torch.optim.Optimizer) -> None:
    expected = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
    actual = [id(parameter) for group in optimizer.param_groups for parameter in group["params"]]
    if set(actual) != expected or len(actual) != len(set(actual)):
        raise RuntimeError("Optimizer tidak mencakup seluruh parameter student/projector tepat sekali.")
    if model.teacher_model.training or any(parameter.requires_grad for parameter in model.teacher_model.parameters()):
        raise RuntimeError("Teacher harus frozen/eval.")
    if set(actual) & {id(parameter) for parameter in model.teacher_model.parameters()}:
        raise RuntimeError("Teacher masuk optimizer.")


class PKDTrainer(DetectionTrainer):
    """Small integration layer; retains native criterion, optimizer math, EMA and GradScaler."""
    def __init__(self, *args: Any, **kwargs: Any):
        if (kwargs.get("overrides") or {}).get("resume"):
            raise ValueError("RESUME_NOT_VERIFIED: pemulihan engine/loader belum didukung oleh notebook ini.")
        super().__init__(*args, **kwargs)
        self._requested_batch = self.args.batch
        self.add_callback("on_train_batch_start", self._strict_batch_start)
        self.add_callback("on_train_batch_end", self._strict_batch_end)

    def set_model_attributes(self) -> None:
        super().set_model_attributes()
        LAST_PHASE["value"] = "initialization"
        if self.args.resume or self.args.compile or self.world_size > 1:
            raise ValueError("PKD pilot: satu GPU, compile=False, resume=False.")
        self.model = PKDModel(self.args.distill_model, self.model)

    def _build_train_pipeline(self) -> None:
        if self.batch_size != self._requested_batch or self.args.batch != self._requested_batch:
            raise RuntimeError("Batch berubah sesudah OOM; protokol PKD tidak boleh diubah otomatis.")
        super()._build_train_pipeline()

    def build_optimizer(self, model: nn.Module, *args: Any, **kwargs: Any):
        optimizer = super().build_optimizer(model, *args, **kwargs)
        for group in optimizer.param_groups:
            group["params"] = [parameter for parameter in group["params"] if parameter.requires_grad]
        audit_optimizer(unwrap_model(model), optimizer)
        return optimizer

    @staticmethod
    def _strict_batch_start(trainer: Any) -> None:
        trainer._oom_retries = 3  # Disable the pinned engine's first-epoch OOM batch-halving path.
        if trainer.batch_size != trainer.args.batch or trainer.args.imgsz != IMGSZ:
            raise RuntimeError("Batch/resolusi efektif berbeda dari protokol.")
        if bool(trainer.amp) != AMP:
            raise RuntimeError("AMP yang diminta tidak aktif; hentikan, jangan fallback diam-diam.")
        LAST_PHASE["value"] = "batch_prepare"

    @staticmethod
    def _strict_batch_end(trainer: Any) -> None:
        trainer._oom_retries = 0
        if not torch.isfinite(trainer.loss):
            raise RuntimeError("Loss training tidak finite.")
        model = unwrap_model(trainer.model)
        if any(parameter.grad is not None for parameter in model.teacher_model.parameters()):
            raise RuntimeError("Gradien bocor ke teacher.")
        if model._teacher_feats or model._student_feats:
            raise RuntimeError("Feature hooks masih menahan tensor setelah loss.")

    def optimizer_step(self) -> None:
        LAST_PHASE["value"] = "optimizer_step"
        # Keep the pinned engine's unscale/clip/step/update/EMA/zero_grad unchanged.
        return super().optimizer_step()

    def validate(self):
        LAST_PHASE["value"] = "validation"
        return super().validate()

    def _handle_nan_recovery(self, epoch: int) -> bool:
        if not torch.isfinite(self.loss) or (self.fitness is not None and not math.isfinite(float(self.fitness))):
            raise RuntimeError("Loss/fitness non-finite; otomatis reload checkpoint dinonaktifkan pada pilot PKD.")
        return False

    def train(self):
        try:
            return super().train()
        finally:
            model = unwrap_model(self.model)
            if isinstance(model, DistillationModel):
                model._remove_feature_hooks()


In [ ]:
def cpu_tree(value: Any) -> Any:
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(cpu_tree(item) for item in value)
    return copy.deepcopy(value)


def training_state(trainer: Any) -> dict[str, Any]:
    """Portable tensor/state archive, not an advertised Ultralytics resume checkpoint."""
    model, ema = unwrap_model(trainer.model), unwrap_model(trainer.ema.ema)
    generator = getattr(trainer.train_loader, "generator", None)
    return {
        "format": "pkd-training-state-v1", "resume_status": RESUME_STATUS,
        "resume_limitations": ["Engine local iteration/accumulation counters and worker prefetch state are not restored/tested"],
        "epoch_zero_based": int(trainer.epoch), "seed": SEED,
        "student": cpu_tree(model.student_model.state_dict()), "projector": cpu_tree(model.projector.state_dict()),
        "ema_student": cpu_tree(ema.student_model.state_dict()), "ema_projector": cpu_tree(ema.projector.state_dict()),
        "ema_updates": int(trainer.ema.updates), "optimizer": cpu_tree(trainer.optimizer.state_dict()),
        "scheduler": cpu_tree(trainer.scheduler.state_dict()), "scaler": cpu_tree(trainer.scaler.state_dict()),
        "pending_gradients": {name: cpu_tree(parameter.grad) for name, parameter in model.named_parameters()
                              if parameter.requires_grad and parameter.grad is not None},
        "criterion_updates": getattr(model.criterion, "updates", None),
        "best_fitness": float(trainer.best_fitness), "fitness": float(trainer.fitness),
        "stopper": cpu_tree(vars(trainer.stopper)), "args": vars(trainer.args).copy(),
        "accumulate": int(trainer.accumulate), "rng_python": random.getstate(),
        "rng_numpy": np.random.get_state(), "rng_torch": torch.get_rng_state(),
        "rng_cuda": [torch.cuda.get_rng_state(trainer.device)] if trainer.device.type == "cuda" else [],
        "loader_generator_state": generator.get_state() if generator is not None else None,
        "teacher_provenance": TEACHER_PROVENANCE, "pkd_weight": PKD_WEIGHT,
        "feature_layout": model.feature_layout, "environment": ENVIRONMENT_INFO,
    }


def export_student(source: Path, destination: Path) -> str:
    """Strip into a new file; never mutate the checkpoint the engine is still training with."""
    if source.resolve() == destination.resolve():
        raise ValueError("Snapshot student harus berbeda dari checkpoint training asli.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = strip_optimizer(source, str(destination), updates={"teacher_provenance": TEACHER_PROVENANCE,
                                                                  "method": METHOD, "resume_status": "INFERENCE_ONLY"})
    model = checkpoint.get("model")
    if not isinstance(model, nn.Module) or isinstance(model, DistillationModel) or hasattr(model, "projector"):
        raise RuntimeError("Export gagal: inference checkpoint masih memerlukan PKD wrapper/projector.")
    if model.yaml.get("scale") != "n":
        raise RuntimeError("Export harus berupa student YOLO26n.")
    for layer in model.modules():
        if any(isinstance(hook, FeatureHook) for hook in layer._forward_hooks.values()):
            raise RuntimeError("Student inference masih memiliki hook KD.")
    del checkpoint, model
    return file_sha256(destination)


def snapshot_epoch(trainer: Any, context: dict[str, Any]) -> dict[str, Any]:
    """Immutable per-epoch snapshots, including class-independent training state archives."""
    epoch = int(trainer.epoch) + 1
    status("CHECKPOINT", f"Epoch {epoch}: menyiapkan snapshot student dan training-state.")
    snapshot = context["directory"] / "epoch_snapshots" / f"epoch-{epoch:03d}"
    snapshot.mkdir(parents=True, exist_ok=False)
    export_student(Path(trainer.last), snapshot / "last.pt")
    export_student(Path(trainer.best), snapshot / "best.pt")
    torch.save(training_state(trainer), snapshot / "training_state_last.pt")
    is_best = float(trainer.fitness) == float(trainer.best_fitness)
    if is_best:
        context["best_epoch"] = epoch
        shutil.copy2(snapshot / "training_state_last.pt", snapshot / "training_state_best.pt")
    elif context.get("best_state_path"):
        shutil.copy2(context["best_state_path"], snapshot / "training_state_best.pt")
    else:
        raise RuntimeError("Tidak ada training-state untuk best epoch; snapshot tidak boleh diberi label best.")
    context["best_state_path"] = snapshot / "training_state_best.pt"
    for name in ("results.csv", "args.yaml"):
        path = Path(trainer.save_dir) / name
        if not path.is_file():
            raise FileNotFoundError(f"File epoch wajib tidak tersedia: {path}")
        shutil.copy2(path, snapshot / name)
    (snapshot / "teacher_provenance.json").write_text(json.dumps(TEACHER_PROVENANCE, indent=2), encoding="utf-8")
    manifest = {path.name: {"sha256": file_sha256(path), "bytes": path.stat().st_size}
                for path in snapshot.iterdir() if path.is_file()}
    (snapshot / "checkpoint_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return {"directory": snapshot, "epoch": epoch, "is_best": is_best, "manifest": manifest}


In [ ]:
def gradient_norm(parameters) -> float:
    values = [parameter.grad.detach().float().norm() for parameter in parameters if parameter.grad is not None]
    if not values:
        raise RuntimeError("Tidak ada gradien pada komponen trainable yang diwajibkan.")
    norm = torch.stack(values).norm()
    if not torch.isfinite(norm) or float(norm) <= 0:
        raise RuntimeError("Gradien student/projector tidak finite atau seluruhnya nol.")
    return float(norm)


def make_callbacks(context: dict[str, Any]) -> dict[str, Any]:
    """Bind every callback to one diagnostic/training run; log each epoch exactly once."""
    def start(trainer):
        model = unwrap_model(trainer.model)
        audit_optimizer(model, trainer.optimizer)
        if trainer.batch_size != context["batch"] or int(trainer.args.imgsz) != IMGSZ or bool(trainer.amp) != AMP:
            raise RuntimeError("Protokol efektif batch/imgsz/AMP berbeda dari konfigurasi.")
        if context["mode"] == "diagnostic" and len(trainer.train_loader) < DIAGNOSTIC_STEPS:
            raise ValueError("Satu epoch train harus menyediakan minimal DIAGNOSTIC_STEPS batch untuk preflight ini.")
        context["teacher_before"] = state_digest(model.teacher_model)
        context["feature_layout"] = model.feature_layout
        context["train_batches_per_epoch"] = len(trainer.train_loader)
        context["run"].config.update({"feature_taps": model.feature_layout, "actual_batch": trainer.batch_size,
                                      "actual_amp": bool(trainer.amp), "effective_training_args": vars(trainer.args),
                                      "resume_status": RESUME_STATUS})
        if trainer.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(trainer.device)
        def before_step(optimizer, args, kwargs):
            # GradScaler calls this ONLY for an actual step, after native unscale + clipping.
            context["gradient"] = {
                "student": gradient_norm(model.student_model.parameters()),
                "projector": gradient_norm(model.projector.parameters()),
            }
        if context["mode"] == "diagnostic":
            context["optimizer_hook"] = trainer.optimizer.register_step_pre_hook(before_step)
        status("TRAIN", f"{context['mode']}: setup selesai, feature taps={model.feature_layout}")

    def batch_start(trainer):
        context["gradient"] = None
        context["scale_before"] = float(trainer.scaler.get_scale())
        if context["mode"] == "diagnostic" and trainer.device.type == "cuda":
            torch.cuda.synchronize(trainer.device)
        context["batch_start"] = time.perf_counter()

    def batch_end(trainer):
        context["batches"] += 1
        model = unwrap_model(trainer.model)
        if model.teacher_model.training or any(parameter.grad is not None for parameter in model.teacher_model.parameters()):
            raise RuntimeError("Teacher menerima gradien atau keluar dari eval mode.")
        if context["mode"] != "diagnostic":
            return
        if trainer.device.type == "cuda":
            torch.cuda.synchronize(trainer.device)
        row = {"step": context["batches"], "iteration_seconds": time.perf_counter() - context["batch_start"],
               "actual_batch": model.last_actual_batch, "imgsz": int(trainer.args.imgsz),
               "loss_finite": bool(torch.isfinite(trainer.loss)),
               "amp_scale_before": context["scale_before"], "amp_scale_after": float(trainer.scaler.get_scale()),
               "optimizer_step_performed": context["gradient"] is not None,
               "gradient_norm_after_native_clipping": context["gradient"],
               "allocated_bytes": torch.cuda.memory_allocated(trainer.device) if trainer.device.type == "cuda" else 0,
               "reserved_bytes": torch.cuda.memory_reserved(trainer.device) if trainer.device.type == "cuda" else 0}
        row.update({key: float(value) for key, value in trainer.loss_items.items()})
        context["feature_shapes"] = copy.deepcopy(model.last_feature_shapes)
        if not row["loss_finite"] or not all(math.isfinite(value) for value in row.values() if isinstance(value, float)):
            raise RuntimeError("Diagnostic mendeteksi loss/metrik non-finite.")
        context["rows"].append(row)
        payload = {f"diagnostic/{key}": value for key, value in row.items() if isinstance(value, (int, float, bool))}
        if context["gradient"]:
            payload.update({f"diagnostic/grad_norm_{key}": value for key, value in context["gradient"].items()})
        context["run"].log(payload, step=context["batches"] - 1)
        if context["batches"] >= DIAGNOSTIC_STEPS:
            context["teacher_unchanged"] = state_digest(model.teacher_model) == context["teacher_before"]
            if not context["teacher_unchanged"]:
                raise RuntimeError("Parameter atau statistik BatchNorm teacher berubah.")
            trainer.stop = True  # Pinned engine validates/saves this partial diagnostic epoch, then exits.

    def save(trainer):
        epoch = int(trainer.epoch) + 1
        if epoch in context["uploaded_epochs"]:
            return
        LAST_PHASE["value"] = "checkpoint"
        record = snapshot_epoch(trainer, context)
        run = context["run"]
        artifact = wandb.Artifact(f"{EXPERIMENT_ID}-{run.id}-checkpoints", type="model",
            metadata={"epoch": epoch, "is_best_epoch": record["is_best"], "best_epoch_so_far": context["best_epoch"],
                      "fitness": float(trainer.fitness), "best_fitness": float(trainer.best_fitness),
                      "seed": SEED, "method": METHOD, "pkd_weight": PKD_WEIGHT,
                      "teacher_provenance": TEACHER_PROVENANCE, "feature_taps": context["feature_layout"],
                      "resume_status": RESUME_STATUS, "diagnostic_only": context["mode"] == "diagnostic",
                      "partial_diagnostic_epoch": context["mode"] == "diagnostic"})
        for path in sorted(record["directory"].iterdir()):
            artifact.add_file(str(path), name=path.name)
        aliases = [f"epoch-{epoch:03d}", "latest"] + (["best"] if record["is_best"] else [])
        try:
            status("W&B", f"Upload checkpoint epoch {epoch}; proses menunggu sampai selesai.")
            run.log_artifact(artifact, aliases=aliases).wait()
        except BaseException:
            (record["directory"] / "upload_status.json").write_text(json.dumps({"status": "FAIL", "epoch": epoch}), encoding="utf-8")
            raise
        (record["directory"] / "upload_status.json").write_text(json.dumps({"status": "PASS", "epoch": epoch}), encoding="utf-8")
        context["uploaded_epochs"].add(epoch)
        if epoch not in context["logged_epochs"]:
            prefix = "diagnostic_epoch" if context["mode"] == "diagnostic" else "train"
            values = {f"{prefix}/{key}": float(value) for key, value in trainer.tloss.items()}
            values.update({key: float(value) for key, value in trainer.metrics.items()})
            values.update({key: float(value) for key, value in trainer.lr.items()})
            values.update(epoch=epoch, **{"checkpoint/best_epoch": context["best_epoch"],
                                         "protocol/batch": int(trainer.batch_size), "protocol/amp": bool(trainer.amp)})
            if trainer.device.type == "cuda":
                values["memory/peak_allocated_bytes"] = torch.cuda.max_memory_allocated(trainer.device)
            run.log(values, step=DIAGNOSTIC_STEPS + epoch if context["mode"] == "diagnostic" else epoch)
            context["logged_epochs"].add(epoch)

    return {"on_train_start": start, "on_train_batch_start": batch_start,
            "on_train_batch_end": batch_end, "on_model_save": save}


In [ ]:
def code_signature(code: types.CodeType) -> dict[str, Any]:
    """Stable semantic code signature, independent of marshal's string-interning flags."""
    def constant(value):
        if isinstance(value, types.CodeType):
            return code_signature(value)
        if isinstance(value, tuple):
            return [constant(item) for item in value]
        if isinstance(value, frozenset):
            return sorted(repr(item) for item in value)
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return repr(value)
    return {"bytecode": code.co_code.hex(), "constants": [constant(value) for value in code.co_consts],
            "names": code.co_names, "variables": code.co_varnames, "freevars": code.co_freevars}


def protocol_evidence() -> dict[str, Any]:
    """Bind a successful diagnostic to current code, effective engine config, teacher and GPU."""
    arguments = vars(get_cfg(overrides=protocol_args(SESSION_DIR / "fingerprint")))
    for name in ("project", "name", "plots", "distill_model"):
        arguments.pop(name, None)
    functions = (pkd_normalize, pkd_feature_loss, neck_taps, teacher_neck_forward,
                 PKDModel.__init__, PKDModel.forward, PKDModel.loss, PKDTrainer.__init__,
                 PKDTrainer.set_model_attributes, PKDTrainer.build_optimizer, PKDTrainer._build_train_pipeline,
                 PKDTrainer._strict_batch_start, PKDTrainer._strict_batch_end, PKDTrainer.optimizer_step,
                 PKDTrainer.validate, PKDTrainer._handle_nan_recovery, protocol_args,
                 gradient_norm, make_callbacks, snapshot_epoch, export_student)
    signatures = [code_signature(function.__code__) for function in functions]
    code_hash = hashlib.sha256(json.dumps(signatures, sort_keys=True).encode()).hexdigest()
    payload = {"args": arguments, "code": code_hash, "pkd_weight": PKD_WEIGHT, "epsilon": PKD_EPS,
               "teacher_sha256": TEACHER_PROVENANCE["best_pt_sha256"], "teacher_artifact": TEACHER_ARTIFACT,
               "dataset_names": DATASET_NAMES, "student": MODEL_WEIGHTS,
               "dataset_yaml_sha256": file_sha256(Path(DATA)) if Path(DATA).is_file() else None,
               "evaluation_split": EVAL_SPLIT,
               "student_sha256": file_sha256(Path(MODEL_WEIGHTS)) if Path(MODEL_WEIGHTS).is_file() else None,
               "torch": torch.__version__, "ultralytics": metadata.version("ultralytics"),
               "gpu": ENVIRONMENT_INFO.get("gpu"), "diagnostic_batch": DIAGNOSTIC_BATCH,
               "diagnostic_steps": DIAGNOSTIC_STEPS, "diagnostic_warmup": DIAGNOSTIC_WARMUP_STEPS}
    return payload


def protocol_fingerprint() -> str:
    return hashlib.sha256(json.dumps(protocol_evidence(), sort_keys=True, default=str).encode()).hexdigest()


def diagnostic_assessment(context: dict[str, Any]) -> dict[str, Any]:
    rows = context["rows"]
    if len(rows) != DIAGNOSTIC_STEPS or not 0 <= DIAGNOSTIC_WARMUP_STEPS < DIAGNOSTIC_STEPS:
        raise RuntimeError("Jumlah langkah/warmup diagnostic tidak memenuhi konfigurasi.")
    measured = rows[DIAGNOSTIC_WARMUP_STEPS:]
    tail = measured[-min(5, len(measured)):]
    if not all(row["optimizer_step_performed"] for row in tail):
        raise RuntimeError("AMP belum stabil pada langkah pengukuran terakhir; diagnostic belum PASS.")
    if any(row["actual_batch"] != BATCH or row["imgsz"] != IMGSZ for row in rows):
        raise RuntimeError("GPU diagnostic tidak memakai batch/resolusi target.")
    width = min(3, len(measured))
    early = statistics.mean(row["allocated_bytes"] for row in measured[:width])
    late = statistics.mean(row["allocated_bytes"] for row in measured[-width:])
    growth = late - early
    if growth > MEMORY_GROWTH_TOL_MB * 1024**2:
        raise RuntimeError(f"Pertumbuhan allocated VRAM setelah warmup terlalu besar: {growth / 1024**2:.1f} MiB.")
    if not context.get("teacher_unchanged"):
        raise RuntimeError("Teacher parameters/BN belum terbukti tetap selama diagnostic.")
    seconds = statistics.median(row["iteration_seconds"] for row in measured)
    return {"status": "PASS", "passed": True, "fingerprint": protocol_fingerprint(),
            "fingerprint_evidence": protocol_evidence(),
            "gpu": ENVIRONMENT_INFO.get("gpu"), "total_vram_bytes": context["total_vram_bytes"],
            "peak_allocated_bytes": context["peak_allocated_bytes"],
            "peak_reserved_bytes": context["peak_reserved_bytes"], "allocated_growth_bytes": growth,
            "actual_batch": BATCH, "imgsz": IMGSZ, "steps": len(rows),
            "warmup_steps": DIAGNOSTIC_WARMUP_STEPS, "median_iteration_seconds": seconds,
            "train_batches_per_epoch": context["train_batches_per_epoch"],
            "compute_only_100epoch_hours_estimate": seconds * context["train_batches_per_epoch"] * EPOCHS / 3600,
            "estimate_excludes": ["validation", "checkpoint export/upload", "setup", "session interruption"],
            "teacher_parameters_and_bn_unchanged": True, "feature_taps": context["feature_layout"],
            "feature_shapes": context.get("feature_shapes"), "rows": rows,
            "amp_backoff_events": sum(row["amp_scale_after"] < row["amp_scale_before"] for row in rows),
            "gradient_norm_definition": "student/projector norm at optimizer pre-step, AFTER native clipping",
            "resume_status": RESUME_STATUS}


def require_full_training(diagnostic: dict[str, Any] | None) -> None:
    require_cuda()
    if not RUN_FULL_TRAINING:
        raise RuntimeError("RUN_FULL_TRAINING harus diaktifkan secara eksplisit setelah diagnostic.")
    if DIAGNOSTIC_BATCH != BATCH:
        raise RuntimeError("Diagnostic batch harus sama dengan batch penelitian.")
    if NUMERICAL_RESULT.get("status") != "PASS" or NUMERICAL_RESULT.get("reference_sha256") != REFERENCE_SHA256:
        raise RuntimeError("Numerical verification terhadap referensi resmi belum PASS.")
    if EVAL_SPLIT != "val":
        raise RuntimeError("Test tidak boleh dipakai untuk tuning atau pemilihan checkpoint.")
    if not diagnostic or diagnostic.get("passed") is not True or diagnostic.get("status") != "PASS":
        raise RuntimeError("Full training diblokir: GPU diagnostic belum PASS.")
    if diagnostic.get("fingerprint") != protocol_fingerprint():
        raise RuntimeError("Konfigurasi/kode/GPU/teacher berubah sejak diagnostic; jalankan diagnostic lagi.")
    if file_sha256(Path(TEACHER_CHECKPOINT)) != TEACHER_PROVENANCE["best_pt_sha256"]:
        raise RuntimeError("Byte checkpoint teacher berubah sejak audit.")


def evaluate_student(best: Path, directory: Path) -> dict[str, Any]:
    model = None
    try:
        model = YOLO(str(best))
        if isinstance(model.model, DistillationModel) or model.model.yaml.get("scale") != "n":
            raise RuntimeError("Evaluasi final memerlukan checkpoint YOLO26n student-only.")
        LAST_PHASE["value"] = "validation"
        metrics = model.val(data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                            plots=True, verbose=False, project=str(directory), name="student_validation")
        try:
            flops = float(get_flops(model.model, imgsz=IMGSZ))
        except (AttributeError, ImportError, RuntimeError, TypeError, ValueError):
            flops = None
        return {"map50_95": float(metrics.box.map), "map50": float(metrics.box.map50),
                "precision": float(metrics.box.mp), "recall": float(metrics.box.mr),
                "ap_per_class": {DATASET_NAMES[int(index)]: float(metrics.box.maps[int(index)])
                                 for index in metrics.box.ap_class_index}, "class_names": DATASET_NAMES,
                "parameters_student": sum(parameter.numel() for parameter in model.model.parameters()),
                "gflops_student": flops, "inference_ms_per_image": float(metrics.speed.get("inference", 0)),
                "eval_split": "val"}
    finally:
        model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_mode(mode: str) -> dict[str, Any]:
    """Run a fresh diagnostic OR fresh full model, with isolated W&B and immutable snapshots."""
    require_cuda()
    if mode not in {"diagnostic", "training"}:
        raise ValueError("Mode harus diagnostic atau training.")
    if mode == "diagnostic":
        globals()["DIAGNOSTIC_RESULT"] = None
    if EVAL_SPLIT != "val":
        raise ValueError("Pilot PKD hanya memakai val; test adalah tahap evaluasi terpisah.")
    if file_sha256(Path(TEACHER_CHECKPOINT)) != TEACHER_PROVENANCE["best_pt_sha256"]:
        raise RuntimeError("Checkpoint teacher berubah setelah audit.")
    if mode == "training":
        require_full_training(globals().get("DIAGNOSTIC_RESULT"))
        status("CHECKPOINT", "RESUME_NOT_VERIFIED: state disimpan, tetapi resume engine/RNG/loader belum diuji. resume=True ditolak.")
    if wandb.run is not None:
        raise RuntimeError("Masih ada run W&B aktif; notebook tidak akan mengambil alih run lain.")
    if DIAGNOSTIC_BATCH != BATCH or DIAGNOSTIC_STEPS < 2:
        raise ValueError("Diagnostic harus memakai batch target dan minimal dua langkah.")
    if not math.isfinite(PKD_WEIGHT) or PKD_WEIGHT <= 0:
        raise ValueError("Pilot diagnostic memerlukan PKD_WEIGHT positif dan finite.")
    directory = SESSION_DIR / f"{mode}_{datetime.now(UTC):%Y%m%dT%H%M%S_%fZ}"
    directory.mkdir(parents=True, exist_ok=False)
    token = datetime.now(UTC).strftime("%Y%m%d_%H%M%S_%f")
    name = f"{EXPERIMENT_ID}_seed{SEED}_{token}" if mode == "training" else f"{EXPERIMENT_ID}_diagnostic_seed{SEED}_{token}"
    run = student = trainer = None
    context = {"directory": directory, "mode": mode, "batch": BATCH, "batches": 0, "rows": [],
               "best_epoch": None, "uploaded_epochs": set(), "logged_epochs": set()}
    started = time.perf_counter()
    try:
        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP,
            name=name, id=wandb.util.generate_id(), resume="never", job_type=mode,
            settings=wandb.Settings(init_timeout=90), tags=[*WANDB_TAGS, mode, f"pkdw{PKD_WEIGHT:g}"],
            config={"experiment": EXPERIMENT_ID, "method": METHOD, "teacher": "yolo26s", "model": MODEL_WEIGHTS,
                    "dataset": DATA, "seed": SEED, "pkd_weight": PKD_WEIGHT, "pkd_epsilon": PKD_EPS,
                    "teacher_artifact": TEACHER_ARTIFACT, "teacher_provenance": TEACHER_PROVENANCE,
                    "environment": ENVIRONMENT_INFO, "resume_status": RESUME_STATUS,
                    "diagnostic_only": mode == "diagnostic", "numerical_reference": NUMERICAL_RESULT})
        context["run"] = run
        status("W&B", f"{mode}: {run.url}")
        linked = run.use_artifact(TEACHER_ARTIFACT, type="model")
        if linked.version != "v62":
            raise ValueError("Run ditautkan ke versi artifact yang tidak sesuai.")
        init_seeds(SEED, deterministic=True)
        LAST_PHASE["value"] = "initialization"
        student = YOLO(MODEL_WEIGHTS)
        for event, callback in make_callbacks(context).items():
            student.add_callback(event, callback)
        student.train(trainer=PKDTrainer, **protocol_args(directory, diagnostic=mode == "diagnostic"))
        trainer = student.trainer
        if trainer.device.type == "cuda":
            context.update(total_vram_bytes=torch.cuda.get_device_properties(trainer.device).total_memory,
                           peak_allocated_bytes=torch.cuda.max_memory_allocated(trainer.device),
                           peak_reserved_bytes=torch.cuda.max_memory_reserved(trainer.device))
        else:
            context.update(total_vram_bytes=0, peak_allocated_bytes=0, peak_reserved_bytes=0)
        if mode == "diagnostic":
            report = diagnostic_assessment(context)
        else:
            best_snapshot = directory / "epoch_snapshots" / f"epoch-{max(context['uploaded_epochs']):03d}" / "best.pt"
            report = {"status": "PASS", **evaluate_student(best_snapshot, directory),
                      "completed_epochs": len(context["uploaded_epochs"]), "best_epoch": context["best_epoch"],
                      "student_checkpoint": str(best_snapshot), "resume_status": RESUME_STATUS}
        report.update(mode=mode, seed=SEED, wandb_run_id=run.id, wandb_url=run.url,
                      run_directory=str(directory), duration_seconds=time.perf_counter() - started)
        path = directory / ("diagnostic_report.json" if mode == "diagnostic" else "metrics.json")
        path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        (directory / "teacher_provenance.json").write_text(json.dumps(TEACHER_PROVENANCE, indent=2), encoding="utf-8")
        (directory / "config.json").write_text(json.dumps({"effective_args": vars(trainer.args),
            "environment": ENVIRONMENT_INFO, "teacher_provenance": TEACHER_PROVENANCE,
            "feature_taps": context["feature_layout"], "pkd_weight": PKD_WEIGHT,
            "reference_url": REFERENCE_URL, "reference_sha256": REFERENCE_SHA256,
            "resume_status": RESUME_STATUS}, indent=2, default=str), encoding="utf-8")
        final_artifact = wandb.Artifact(f"{EXPERIMENT_ID}-{run.id}-results", type="results",
                                       metadata={"mode": mode, "seed": SEED, "resume_status": RESUME_STATUS})
        for filename in (path.name, "teacher_provenance.json", "config.json"):
            final_artifact.add_file(str(directory / filename), name=filename)
        for plot in sorted(Path(trainer.save_dir).glob("*.png")):
            run.log({f"plots/{plot.stem}": wandb.Image(str(plot))})
            final_artifact.add_file(str(plot), name=f"plots/{plot.name}")
        run.log_artifact(final_artifact, aliases=["final"]).wait()
        run.summary.update({f"final/{key}": value for key, value in report.items()
                            if isinstance(value, (int, float, str, bool))})
        run.finish()
        return report
    except BaseException as error:
        message = {"status": "FAIL", "mode": mode, "phase": LAST_PHASE["value"],
                   "error_type": type(error).__name__, "message": str(error),
                   "batch": BATCH, "imgsz": IMGSZ, "amp": AMP,
                   "protocol_changed_automatically": False}
        (directory / "failure.json").write_text(json.dumps(message, indent=2), encoding="utf-8")
        status("FAIL", f"Fase {LAST_PHASE['value']}: {type(error).__name__}: {error}")
        if "out of memory" in str(error).lower():
            status("OOM", "Antrean dihentikan. Opsi berikutnya adalah meninjau penggunaan VRAM atau mendokumentasikan perubahan batch/resolusi secara eksplisit; notebook tidak mengubah protokol.")
        if run is not None:
            try:
                run.finish(exit_code=1)
            except Exception:  # noqa: BLE001 -- Preserve the original error even when W&B finish also fails.
                status("W&B", "Penutupan run gagal; file lokal dan snapshot tidak dihapus.")
        raise
    finally:
        hook = context.get("optimizer_hook")
        if hook is not None:
            hook.remove()
        if student is not None and hasattr(student, "trainer"):
            for loader in (getattr(student.trainer, "train_loader", None), getattr(student.trainer, "test_loader", None)):
                if hasattr(loader, "close"):
                    loader.close()
        student = trainer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
DIAGNOSTIC_RESULT = None
DIAGNOSTIC_RESULT = run_mode("diagnostic")
print(json.dumps({key: value for key, value in DIAGNOSTIC_RESULT.items() if key != "rows"}, indent=2))


## 13. Ringkasan VRAM dan durasi

Pertumbuhan allocated VRAM setelah warmup diperiksa; reserved dilaporkan sebagai informasi allocator. Estimasi berikut hanya komputasi batch dan tidak menjamin sesi Kaggle cukup untuk 100 epoch.


In [ ]:
status("DIAGNOSTIC", f"{DIAGNOSTIC_RESULT['status']}; peak allocated={DIAGNOSTIC_RESULT['peak_allocated_bytes'] / 1024**3:.2f} GiB, peak reserved={DIAGNOSTIC_RESULT['peak_reserved_bytes'] / 1024**3:.2f} GiB")
status("ESTIMATE", f"Estimasi komputasi saja {DIAGNOSTIC_RESULT['compute_only_100epoch_hours_estimate']:.2f} jam; belum termasuk validasi/upload/setup. Ini estimasi dari diagnostic, bukan waktu training yang telah diukur.")
status("CHECKPOINT", RESUME_STATUS + ": state tersedia untuk audit/pemulihan lanjutan; auto-resume belum didukung.")


## 14. Full training dengan gate eksplisit

Set RUN_FULL_TRAINING=True hanya setelah meninjau diagnostic PASS, rasio loss, VRAM, dan keterbatasan resume. Boleh menjalankan ulang cell ini setelah mengganti flag di konfigurasi, selama fingerprint masih cocok. Jika kode/konfigurasi/teacher berubah, diagnostic wajib diulang. Run All dengan True tetap melakukan diagnostic baru dahulu.


In [ ]:
FULL_RESULT = None
if RUN_FULL_TRAINING:
    require_full_training(DIAGNOSTIC_RESULT)
    FULL_RESULT = run_mode("training")
else:
    status("GATE", "Diagnostic selesai. Tinjau loss, rasio PKD/detection, gradien, VRAM dan status resume sebelum mengaktifkan RUN_FULL_TRAINING=True.")


## 15. Perlindungan checkpoint per epoch

Setiap epoch yang selesai memiliki snapshot unik: student-only best.pt/last.pt, training_state_last.pt/training_state_best.pt (tensor/state archive), results.csv, args.yaml, provenance dan manifest SHA-256. Upload W&B ditunggu sampai selesai. upload_status.json lokal mencatat PASS/FAIL; snapshot tidak dihapus jika upload gagal. Checkpoint engine mentah tetap lokal dan tidak dipublikasikan sebagai best.pt wrapper.

**RESUME_NOT_VERIFIED:** optimizer/scheduler/scaler/RNG dicatat, tetapi worker prefetch dan counter lokal loop/akumulasi belum dipulihkan/diuji. Tidak tersedia auto-resume; jangan menjalankan YOLO(last.pt).train(resume=True) untuk mengklaim kelanjutan PKD yang setara.


## 16. Evaluasi validation

Full mode mengevaluasi ulang student-only best.pt pada val dengan protokol batch/imgsz yang sama dengan baseline. mAP50-95 utama, mAP50/precision/recall/AP kelas dan biaya inferensi dicatat. Evaluasi final dilakukan dalam full run sebelum upload hasil dan finish. Tidak ada angka full training jika hanya diagnostic dijalankan.


## 17. Hasil dan metadata

Diagnostic dan training memiliki run ID baru, job_type berbeda, tag berbeda dan direktori output berbeda. Group tetap teacher_s_student_n_pkd. Metrik epoch hanya dicatat sekali; epoch diagnostic parsial diberi label eksplisit. Seed, versi, artifact/SHA teacher, tap, adapter, bobot loss, GPU, durasi dan protokol efektif tersimpan dalam config/artifact.


In [ ]:
if FULL_RESULT is not None:
    print(json.dumps(FULL_RESULT, indent=2))
else:
    print("Belum ada metrik full training. GPU diagnostic tidak membuktikan peningkatan mAP.")
print("Output:", SESSION_DIR)
print("Student-only: best.pt/last.pt di epoch_snapshots. Training archive: training_state_*.pt, RESUME_NOT_VERIFIED.")


## 18. Reproducibility dan batas pengujian

Notebook 01–11 dan dataset tidak diubah. Hasil CPU sintetis tidak membuktikan stabilitas GPU batch16/640 atau keunggulan PKD. GPU/VRAM/waktu hanya boleh dilaporkan dari diagnostic yang benar-benar dijalankan. PKD_WEIGHT memerlukan pemeriksaan skala; jangan menyalin bobot Native 6.0. Native/CrossKD referensi yang dibandingkan adalah konfigurasi s→n, dengan teacher yang sama.

Jika OOM, lihat failure.json untuk fase initialization/teacher_forward/student_forward/pkd_forward/backward/optimizer_step/validation/checkpoint. Run dihentikan tanpa retry batch yang berubah. Full training tidak dimulai setelah diagnostic gagal. Jika sesi Kaggle terputus mendadak, epoch terakhir yang sukses diunggah tetap tersedia di W&B; epoch yang sedang berjalan dapat hilang. Resume engine tetap NOT VERIFIED.
